<a href="https://colab.research.google.com/github/rija-ansari/MSE1003H_RijaAnsari/blob/main/Assignment_4/MSE1003_Assignment4_RA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Comment out the below cells when running in Colab


In [ ]:
#!git clone https://github.com/rija-ansari/MSE1003H_RijaAnsari/

In [ ]:
#import os
#repo = "MSE1003H_RijaAnsari"
#print(os.listdir(repo))


In [ ]:
#%cd MSE1003H_RijaAnsari/Assignment_4

# Assignment 4: Multi-Objective Optimization and Inference


The experiment uses a Britton-Robinson buffer, so I have included the four solution volumes in the data for your reference, along with the measured pH. In practice, you would want to re-derive the HH equation to reflect the different acid properties. However for this experiment, it is fine to just use the volume ratio of all acids to the NaOH base, rather than adjusting the HH equation for all three.


In [ ]:
import os
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error, max_error, mean_absolute_percentage_error

from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, RationalQuadratic, ConstantKernel

from scipy.stats import norm

In [ ]:
#install botorch quietly
!pip install botorch -q

In [ ]:
# ─── Run this first to check your BoTorch version ─────────────────────────────
import botorch, gpytorch, torch
print(f"BoTorch:  {botorch.__version__}")
print(f"GPyTorch: {gpytorch.__version__}")
print(f"PyTorch:  {torch.__version__}")

In [ ]:
import torch
import gpytorch
from gpytorch.means import ConstantMean, MultitaskMean
from gpytorch.kernels import RBFKernel, ScaleKernel, IndexKernel, MultitaskKernel
from gpytorch.distributions import MultitaskMultivariateNormal

from gpytorch.models import ExactGP
from gpytorch.likelihoods import MultitaskGaussianLikelihood
from gpytorch.distributions import MultitaskMultivariateNormal
from gpytorch.means import MultitaskMean, ConstantMean
from gpytorch.kernels import ScaleKernel, RBFKernel, MultitaskKernel

from botorch.models.model import Model
from botorch.posteriors.gpytorch import GPyTorchPosterior
from botorch.acquisition.objective import GenericMCObjective
from botorch.acquisition.monte_carlo import qNoisyExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.optim import optimize_acqf

## Surrogate Models

### for alpha

use Henderson to determine value of alpha and beta

pH = alpha + beta log (Va/Vb)

2 objective functions 
but 1 gpr predict thing
and model thing 

but 2 models in the tqdm 

Got it — this is the real active‑learning situation:
you don’t know α and β yet, and you’re sampling pH values until your surrogate model’s posterior over α and β converges.

That means you cannot train a GP directly on α and β (because they’re not observed).
Instead, you must train a GP on pH, and infer α and β as latent parameters.

This is a classic latent‑parameter inference problem, and there are two clean ways to do it in a Bayesian optimization loop.

Let me lay out the correct structure so your notebook is scientifically tight and your code is reproducible.

In [ ]:
random_seed = 1003

#read the data
data = pd.read_csv("combined_results.csv")
data

In [ ]:
#set up valid data with value of 0.00001 for vol_sodiumHydroxide for zeros
valid_data = data.copy()

#drop sodium hydroxide zeros
valid_data = valid_data[valid_data['vol_sodiumHydroxide'] > 0].reset_index(drop=True)
#valid_data.loc[0, 'vol_sodiumHydroxide'] = 0.00001
valid_data


In [ ]:
valid_data['vol_acid'] = (valid_data['vol_aceticAcid'] + 
                          valid_data['vol_phosphoricAcid'] + 
                          valid_data['vol_boricAcid'])
valid_data['vol_base']  = valid_data['vol_sodiumHydroxide'].replace(0, 1e-6)
valid_data['vol_ratio'] = valid_data['vol_acid'] / valid_data['vol_base']

valid_data

In [ ]:
# ─── Objective functions ────────────────────────────────────────────────

def objective_alpha(log10_ratio, train_y):
    """
    log10_ratio : (n,) or (n,1) tensor of log10(vol_ratio) — already log-scaled
    train_y     : (n,)   tensor of pH values
    """
    log10_ratio = log10_ratio.squeeze()
    X     = torch.stack([torch.ones_like(log10_ratio), log10_ratio], dim=1)
    theta = torch.linalg.lstsq(X, train_y.unsqueeze(1)).solution
    return theta[0].item()

def objective_beta(log10_ratio, train_y):
    log10_ratio = log10_ratio.squeeze()
    X     = torch.stack([torch.ones_like(log10_ratio), log10_ratio], dim=1)
    theta = torch.linalg.lstsq(X, train_y.unsqueeze(1)).solution
    return theta[1].item()

def get_alpha_beta_per_point(log10_ratio, train_y):
    """
    log10_ratio : (n, 1) tensor — already log10 scaled, do NOT log again
    train_y     : (n,)   tensor of pH values
    Returns     : (n, 2) tensor of (alpha, beta) LOO estimates
    """
    n = len(log10_ratio)
    results = []
    for i in range(n):
        idx   = [j for j in range(n) if j != i]
        x_sub = log10_ratio[idx]
        y_sub = train_y[idx]
        a = objective_alpha(x_sub, y_sub)
        b = objective_beta(x_sub, y_sub)
        results.append([a, b])
    return torch.tensor(results, dtype=torch.double)

In [ ]:
# ─── Rebuild training tensors cleanly ─────────────────────────────────────────
# Ensure no NaOH zero values are present
valid_data_fit = valid_data[valid_data['vol_sodiumHydroxide'] > 0].copy()
valid_data_fit['log10_vol_ratio'] = np.log10(valid_data_fit['vol_acid'] / valid_data_fit['vol_sodiumHydroxide'])

print("Fitting data:")
print(valid_data_fit[['pH_measured', 'vol_acid', 'vol_sodiumHydroxide', 
                       'log10_vol_ratio', 'epit']])

train_x_joint = torch.tensor(valid_data_fit[['log10_vol_ratio']].values, dtype=torch.double)   # shape (n, 1), values should be in roughly [-1, 2]

train_y_ph = torch.tensor(valid_data_fit['pH_measured'].values, dtype=torch.double)

# LOO estimates — now correctly uses pre-computed log10 values
train_y_ab = get_alpha_beta_per_point(train_x_joint, train_y_ph)

train_y_epit = torch.tensor(valid_data_fit['epit'].values, dtype=torch.double)

train_y_joint = torch.stack([
    train_y_ab[:, 0],
    train_y_ab[:, 1],
    train_y_epit
], dim=1)   # shape (n, 3)

# Sanity check — nothing should be NaN
print(f"\ntrain_x_joint: {train_x_joint.T}")
print(f"train_y_joint:\n{train_y_joint}")
print(f"\nNaN in x: {train_x_joint.isnan().any().item()}")
print(f"NaN in y: {train_y_joint.isnan().any().item()}")

# Bounds on log10 scale — sensible headroom around observed range
log_min = float(train_x_joint.min()) - 1.0
log_max = float(train_x_joint.max()) + 1.0
bounds  = torch.tensor([[log_min], [log_max]], dtype=torch.double)
print(f"\nbounds: log10(ratio) ∈ [{log_min:.3f}, {log_max:.3f}]")
print(f"  i.e. vol_ratio  ∈ [{10**log_min:.3f}, {10**log_max:.3f}]")

In [ ]:
class JointGP(ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)

        self.mean_module = MultitaskMean(ConstantMean(), num_tasks=3)

        self.covar_module = MultitaskKernel(
            ScaleKernel(RBFKernel()),
            num_tasks=3,
            rank=1
        )

    def forward(self, x):
        mean_x  = self.mean_module(x)
        covar_x = self.covar_module(x)
        return MultitaskMultivariateNormal(mean_x, covar_x)

In [ ]:
# ─── Retrain once NaN check passes ───────────────────────────────────────────
likelihood_joint = MultitaskGaussianLikelihood(num_tasks=3).double()
model_joint      = JointGP(train_x_joint, train_y_joint, likelihood_joint).double()

model_joint.train()
likelihood_joint.train()

optimizer = torch.optim.Adam(model_joint.parameters(), lr=0.1)
mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood_joint, model_joint)

losses = []
for i in range(300):
    optimizer.zero_grad()
    loss = -mll(model_joint(train_x_joint), train_y_joint)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (i + 1) % 50 == 0:
        print(f"Iter {i+1}/300  Loss: {loss.item():.4f}")

## Visualization of Surrogate Model Before Acquisition 

In [ ]:
# ─── Reference point ──────────────────────────────────────────────────────────
# Must be worse than the worst observed value on every objective.
# EHVI measures hypervolume improvement RELATIVE to this point —
# setting it too tight (close to observed values) shrinks the
# hypervolume and makes the signal weak.
# A safe rule: observed_min - 10% of observed range per objective.

def make_ref_point(train_y, slack=0.1):
    """
    train_y : (n, 3) tensor — columns are [α, β, E_pitt]
    Returns a (3,) tensor, one ref value per objective.
    All objectives are treated as MAXIMISATION.
    """
    y_min   = train_y.min(dim=0).values
    y_range = train_y.max(dim=0).values - y_min
    return (y_min - slack * y_range).double()

ref_point = make_ref_point(train_y_joint)
print(f"Reference point — α: {ref_point[0]:.4f}, β: {ref_point[1]:.4f}, "
      f"E_pitt: {ref_point[2]:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from botorch.utils.multi_objective.pareto import is_non_dominated

# ─── Existing data only ───────────────────────────────────────────────────────
existing_x = train_x_joint.squeeze().numpy()   # log10(ratio), shape (4,)
existing_y = train_y_joint.numpy()             # (4, 3): [α, β, E_pitt]
n          = len(existing_x)

pH_labels  = [f"pt{int(valid_data_fit['cell'].iloc[i])}\npH={valid_data_fit['pH_measured'].iloc[i]:.1f}"
              for i in range(n)]

# Pareto check
is_pareto  = is_non_dominated(train_y_joint).numpy()   # (4,) bool

face_c = ['#2196F3'] * n
edge_c = ['#2196F3'] * n

# ─── Figure ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
fig.suptitle("Existing Data — Input & Response Space with Pareto Frontier",
             fontsize=14, fontweight='bold', y=0.99)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

ax_input = fig.add_subplot(gs[0, :])          # full-width input space
ax_ab    = fig.add_subplot(gs[1, 0])          # α vs β
ax_aep   = fig.add_subplot(gs[1, 1])          # α vs E_pitt
ax_bep   = fig.add_subplot(gs[1, 2])          # β vs E_pitt

# ─── Panel 1: Input space with GP posteriors ──────────────────────────────────
model_joint.eval()
likelihood_joint.eval()

plot_x = torch.linspace(float(bounds[0]), float(bounds[1]), 300,
                         dtype=torch.double).unsqueeze(1)
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred       = likelihood_joint(model_joint(plot_x))
    plot_mu    = pred.mean.numpy()                   # (300, 3)
    lo, hi     = pred.confidence_region()
    lo, hi     = lo.numpy(), hi.numpy()

plot_x_np = plot_x.squeeze().numpy()

def norm01(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

obj_labels = ['α (normalised)', 'β (normalised)', 'E_pitt (normalised)']
obj_colors = ['#2196F3', '#4CAF50', '#FF9800']
obj_ls     = ['-', '--', '-.']

for t, (lbl, col, ls) in enumerate(zip(obj_labels, obj_colors, obj_ls)):
    mu_n = norm01(plot_mu[:, t])
    lo_n = norm01(lo[:, t])
    hi_n = norm01(hi[:, t])
    ax_input.plot(plot_x_np, mu_n, color=col, lw=2, ls=ls, label=lbl, alpha=0.85)
    ax_input.fill_between(plot_x_np, lo_n, hi_n, alpha=0.12, color=col)

# Plot each existing point
for i in range(n):
    xi = existing_x[i]
    yi = norm01(plot_mu[:, 2])[np.argmin(np.abs(plot_x_np - xi))]
    ax_input.scatter(xi, yi, c=face_c[i], edgecolors=edge_c[i],
                     marker='o', s=130, linewidths=2, zorder=6)
    if is_pareto[i]:
        ax_input.scatter(xi, yi, marker='*', s=80, c='gold',
                         edgecolors='k', linewidths=0.5, zorder=7)
    offset_y = 0.13 if i % 2 == 0 else -0.17
    ax_input.annotate(pH_labels[i],
                      xy=(xi, yi),
                      xytext=(xi + 0.05, yi + offset_y),
                      fontsize=9, ha='center',
                      arrowprops=dict(arrowstyle='->', color='gray', lw=1),
                      bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75))

ax_input.set_xlabel("log₁₀(V_acid / V_base)", fontsize=11)
ax_input.set_ylabel("Normalised objective value", fontsize=11)
ax_input.set_title("Input Feature Space", fontsize=12, fontweight='bold')
ax_input.set_xlim([-1.75, 1.75])
ax_input.set_ylim([-0.3, 1.4])
ax_input.legend(fontsize=9, loc='upper right')
ax_input.grid(True, alpha=0.25)

# ─── Helper: response space panel ─────────────────────────────────────────────
def response_panel(ax, x_idx, y_idx, xlabel, ylabel, title):
    ax.set_title(title, fontsize=11, fontweight='bold')

    # Faded dominated, full opacity non-dominated
    for i in range(n):
        alpha = 1.0 if is_pareto[i] else 0.40
        ax.scatter(existing_y[i, x_idx], existing_y[i, y_idx],
                   c=face_c[i], edgecolors=edge_c[i],
                   marker='o', s=120, linewidths=2,
                   alpha=alpha, zorder=5)
        if is_pareto[i]:
            ax.scatter(existing_y[i, x_idx], existing_y[i, y_idx],
                       marker='*', s=70, c='gold',
                       edgecolors='k', linewidths=0.5, zorder=7)
        x_range = existing_y[:, x_idx].max() - existing_y[:, x_idx].min()
        y_range = existing_y[:, y_idx].max() - existing_y[:, y_idx].min()
        ax.annotate(pH_labels[i],
                    xy=(existing_y[i, x_idx], existing_y[i, y_idx]),
                    xytext=(existing_y[i, x_idx] + 0.03 * x_range,
                            existing_y[i, y_idx] + 0.05 * y_range),
                    fontsize=8, ha='left',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

    # Pareto staircase
    pareto_pts = existing_y[is_pareto][:, [x_idx, y_idx]]
    if len(pareto_pts) > 1:
        sorted_p = pareto_pts[np.argsort(pareto_pts[:, 0])]
        ax.step(sorted_p[:, 0], sorted_p[:, 1], where='post',
                color='gold', lw=2, ls='--', alpha=0.85,
                zorder=4, label='Pareto front')
        ax.legend(fontsize=8)

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.25)

response_panel(ax_ab,  0, 1, 'α',  'β',         'Response: α vs β')
response_panel(ax_aep, 0, 2, 'α',  'E_pitt (V)', 'Response: α vs E_pitt')
response_panel(ax_bep, 1, 2, 'β',  'E_pitt (V)', 'Response: β vs E_pitt')

plt.savefig("pareto_existing_points.png", dpi=150, bbox_inches='tight')
plt.show()
#print("Saved → pareto_existing_points.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.multi_objective.hypervolume import Hypervolume

# ─── LOO performance metrics ──────────────────────────────────────────────────
# With 4 points we simulate "active learning iterations" via LOO:
# iteration k = model trained on k points (added in order of collection)
# This shows how accuracy and uncertainty evolve as data accumulates.

n          = len(train_x_joint)
obj_names  = ['α', 'β', 'E_pitt']
obj_colors = ['#2196F3', '#4CAF50', '#FF9800']

# Storage
rmse_per_iter   = {name: [] for name in obj_names}   # (n_iters,)
std_per_iter    = {name: [] for name in obj_names}    # mean posterior std
hv_per_iter     = []                                  # hypervolume at each iter
iter_labels     = []

# Reference point for hypervolume (must be worse than all observed values)
ref_pt = make_ref_point(train_y_joint, slack=0.1)
hv_calc = Hypervolume(ref_point=ref_pt)

# Simulate iterations: train on first k points, predict on held-out remainder
for k in range(2, n + 1):   # start at 2 — need at least 2 pts for lstsq
    idx_train = list(range(k))
    idx_test  = list(range(k, n))

    x_tr = train_x_joint[idx_train]   # (k, 1)
    y_tr = train_y_joint[idx_train]   # (k, 3)
    x_te = train_x_joint[idx_test]    # (n-k, 1) — empty at last iter
    y_te = train_y_joint[idx_test]    # (n-k, 3)

    # Retrain GP on k points
    lik_k   = MultitaskGaussianLikelihood(num_tasks=3).double()
    model_k = JointGP(x_tr, y_tr, lik_k).double()
    model_k.train(); lik_k.train()
    opt_k = torch.optim.Adam(model_k.parameters(), lr=0.1)
    mll_k = gpytorch.mlls.ExactMarginalLogLikelihood(lik_k, model_k)
    for _ in range(300):
        opt_k.zero_grad()
        loss_k = -mll_k(model_k(x_tr), y_tr)
        loss_k.backward()
        opt_k.step()

    model_k.eval(); lik_k.eval()

    # ── Predictive accuracy on held-out points ────────────────────────────────
    if len(idx_test) > 0:
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred_te = lik_k(model_k(x_te))
            mu_te   = pred_te.mean.numpy()     # (n-k, 3)
        y_te_np = y_te.numpy()
        for t, name in enumerate(obj_names):
            rmse = np.sqrt(np.mean((mu_te[:, t] - y_te_np[:, t])**2))
            rmse_per_iter[name].append(rmse)
    else:
        # Last iter: no held-out points — use train residuals as proxy
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred_tr = lik_k(model_k(x_tr))
            mu_tr   = pred_tr.mean.numpy()
        y_tr_np = y_tr.numpy()
        for t, name in enumerate(obj_names):
            rmse = np.sqrt(np.mean((mu_tr[:, t] - y_tr_np[:, t])**2))
            rmse_per_iter[name].append(rmse)

    # ── Uncertainty: mean posterior std across dense grid ────────────────────
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred_grid = lik_k(model_k(plot_x))
        std_grid  = pred_grid.variance.sqrt().numpy()   # (300, 3)
    for t, name in enumerate(obj_names):
        std_per_iter[name].append(std_grid[:, t].mean())

    # ── Hypervolume of non-dominated set so far ───────────────────────────────
    y_tr_t    = train_y_joint[idx_train]
    pareto_k  = is_non_dominated(y_tr_t)
    pareto_pts = y_tr_t[pareto_k]
    hv        = hv_calc.compute(pareto_pts)
    hv_per_iter.append(hv)

    iter_labels.append(f"k={k}")
    print(f"k={k}: RMSE α={rmse_per_iter['α'][-1]:.4f}  "
          f"β={rmse_per_iter['β'][-1]:.4f}  "
          f"E_pitt={rmse_per_iter['E_pitt'][-1]:.4f}  "
          f"HV={hv:.6f}")

iter_x = np.arange(len(iter_labels))

# ─── Figure 1: Predictive Accuracy ───────────────────────────────────────────
fig1, axes1 = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)
fig1.suptitle("Predictive Accuracy over Active-Learning Iterations\n"
              "(LOO simulation — trained on first k points, tested on remainder)",
              fontsize=13, fontweight='bold')

for t, (name, color) in enumerate(zip(obj_names, obj_colors)):
    ax = axes1[t]
    ax.plot(iter_x, rmse_per_iter[name], 'o-',
            color=color, lw=2.5, ms=9, markeredgecolor='k', markeredgewidth=0.8)
    for xi, yi in zip(iter_x, rmse_per_iter[name]):
        ax.annotate(f"{yi:.4f}", (xi, yi),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=8.5,
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    ax.set_xticks(iter_x)
    ax.set_xticklabels(iter_labels, fontsize=10)
    ax.set_xlabel("Training set size (k)", fontsize=11)
    ax.set_ylabel("RMSE", fontsize=11)
    ax.set_title(f"Surrogate: {name}", fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    # Shade last point (full data — train residual proxy)
    ax.axvspan(iter_x[-1] - 0.4, iter_x[-1] + 0.4,
               alpha=0.12, color='gray', label='Train residual\n(no held-out)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("model_accuracy.png", dpi=150, bbox_inches='tight')
plt.show()

# ─── Figure 2: Uncertainty Evolution + Hypervolume ───────────────────────────
fig2, axes2 = plt.subplots(1, 4, figsize=(18, 4.5))
fig2.suptitle("Uncertainty Evolution & Hypervolume Improvement over Iterations",
              fontsize=13, fontweight='bold')

# Uncertainty panels (one per objective)
for t, (name, color) in enumerate(zip(obj_names, obj_colors)):
    ax = axes2[t]
    ax.plot(iter_x, std_per_iter[name], 's-',
            color=color, lw=2.5, ms=9, markeredgecolor='k', markeredgewidth=0.8)
    for xi, yi in zip(iter_x, std_per_iter[name]):
        ax.annotate(f"{yi:.4f}", (xi, yi),
                    textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=8.5,
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    ax.set_xticks(iter_x)
    ax.set_xticklabels(iter_labels, fontsize=10)
    ax.set_xlabel("Training set size (k)", fontsize=11)
    ax.set_ylabel("Mean posterior σ", fontsize=11)
    ax.set_title(f"Uncertainty: {name}", fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Target: E_pitt should flatten (low std variance across iters)
    # α, β should converge (decreasing std)
    goal = ("Goal: ↓ converge" if name in ['α', 'β']
            else "Goal: flatten (uniform σ)")
    ax.text(0.05, 0.92, goal, transform=ax.transAxes,
            fontsize=8, color=color, style='italic',
            bbox=dict(boxstyle='round', fc='white', alpha=0.6))

# Hypervolume panel (extra credit)
ax = axes2[3]
hv_arr = np.array(hv_per_iter)
hv_improvement = np.diff(hv_arr, prepend=0.0)

ax.bar(iter_x, hv_arr, color='#9C27B0', alpha=0.6,
       edgecolor='k', linewidth=0.8, label='Cumulative HV')
ax.plot(iter_x, hv_arr, 'o-', color='#4A148C',
        lw=2, ms=8, markeredgecolor='k', markeredgewidth=0.8)

ax2_twin = ax.twinx()
ax2_twin.bar(iter_x, hv_improvement, color='#FF5722', alpha=0.35,
             edgecolor='k', linewidth=0.8, label='HV improvement')
ax2_twin.set_ylabel("HV improvement (Δ)", fontsize=10, color='#FF5722')
ax2_twin.tick_params(axis='y', labelcolor='#FF5722')

for xi, yi in zip(iter_x, hv_arr):
    ax.annotate(f"{yi:.4f}", (xi, yi),
                textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

ax.set_xticks(iter_x)
ax.set_xticklabels(iter_labels, fontsize=10)
ax.set_xlabel("Training set size (k)", fontsize=11)
ax.set_ylabel("Hypervolume", fontsize=11)
ax.set_title("Hypervolume Improvement", fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc = 'upper left')

In [ ]:
# ─── Sample from GP posterior to get distribution of α and β predictions ──────
# Draw N samples from the trained GP at each training point and across
# a dense grid — the spread of these samples shows how uncertain the
# model is about α and β, and whether they are converging to a tight range.

N_SAMPLES  = 2000
torch.manual_seed(random_seed)

model_joint.eval()
likelihood_joint.eval()

# Dense grid across input range for sampling
sample_x = torch.linspace(float(bounds[0]), float(bounds[1]),
                           100, dtype=torch.double).unsqueeze(1)   # (100, 1)

# Draw posterior samples — shape (100, N_SAMPLES, 3)
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    post        = model_joint(sample_x)
    samples     = post.sample(torch.Size([N_SAMPLES]))   # (N_SAMPLES, 100, 3)

alpha_samples = samples[:, :, 0].numpy().ravel()   # (N_SAMPLES * 100,)
beta_samples  = samples[:, :, 1].numpy().ravel()

# Global HH fit values for reference lines
alpha_global = objective_alpha(train_x_joint, train_y_ph)
beta_global  = objective_beta(train_x_joint,  train_y_ph)

# LOO estimates (the 4 training targets) for reference
alpha_loo = train_y_joint[:, 0].numpy()
beta_loo  = train_y_joint[:, 1].numpy()

# ─── Figure ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Histogram of Predicted α and β — Convergence Assessment\n"
             f"({N_SAMPLES} posterior samples × 100 input locations)",
             fontsize=13, fontweight='bold')

for ax, samples_arr, loo_vals, global_val, name, color in zip(
    axes,
    [alpha_samples, beta_samples],
    [alpha_loo,     beta_loo],
    [alpha_global,  beta_global],
    ['α (intercept)', 'β (slope)'],
    ['#2196F3',        '#4CAF50']
):
    # Main histogram
    ax.hist(samples_arr, bins=60, color=color, edgecolor='white',
            alpha=0.75, density=True, label='GP posterior samples')

    # Overlay KDE
    from scipy.stats import gaussian_kde
    kde  = gaussian_kde(samples_arr)
    xs   = np.linspace(samples_arr.min(), samples_arr.max(), 300)
    ax.plot(xs, kde(xs), color=color, lw=2.5, label='KDE')

    # Global fit reference line
    ax.axvline(global_val, color='red', lw=2.5, ls='-',
               label=f'Global fit = {global_val:.4f}')

    # LOO training point estimates
    for i, v in enumerate(loo_vals):
        ax.axvline(v, color='orange', lw=1.5, ls='--',
                   label='LOO estimates' if i == 0 else '')
        ax.text(v, ax.get_ylim()[1] * 0.02, f' pt{i+1}',
                color='darkorange', fontsize=8, rotation=90, va='bottom')

    # Convergence stats
    mu_s  = samples_arr.mean()
    std_s = samples_arr.std()
    ax.axvline(mu_s, color='black', lw=1.5, ls=':',
               label=f'Posterior mean = {mu_s:.4f}')
    ax.axvspan(mu_s - std_s, mu_s + std_s,
               alpha=0.12, color='black', label=f'±1σ = {std_s:.4f}')

    ax.set_xlabel(name, fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(f"Distribution of {name} predictions\n"
                 f"mean={mu_s:.4f}, std={std_s:.4f}", fontsize=11)
    ax.legend(fontsize=8.5, loc='upper right')
    ax.grid(True, alpha=0.25)

    # Convergence annotation
    cv = abs(std_s / mu_s) * 100   # coefficient of variation %
    converged = cv < 5.0
    status = f"CV = {cv:.1f}% → {'converged ✓' if converged else 'not yet converged'}"
    ax.text(0.03, 0.97, status, transform=ax.transAxes,
            fontsize=9, va='top',
            color='green' if converged else 'red',
            bbox=dict(boxstyle='round', fc='white', alpha=0.8))

# Fix y-limits after all lines drawn (axvline changes ylim dynamically)
for ax in axes:
    ax.relim()
    ax.autoscale_view()

plt.tight_layout()
plt.savefig("alpha_beta_histograms.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → alpha_beta_histograms.png")

In [ ]:
# ─── Sample α and β from GP posterior ────────────────────────────────────────
# Draw N samples of (α, β) from the joint GP posterior across the input range,
# then compute pH = α + β·log10(r) for each sample to get a distribution
# of HH curves — mean and std of this bundle is what we plot.

N_SAMPLES = 2000
torch.manual_seed(random_seed)

model_joint.eval()
likelihood_joint.eval()

# Sample at a dense grid of log10(ratio) values
sample_x = torch.linspace(float(bounds[0]), float(bounds[1]),
                           200, dtype=torch.double).unsqueeze(1)   # (200, 1)

with torch.no_grad():
    post    = model_joint(sample_x)
    samples = post.sample(torch.Size([N_SAMPLES]))   # (N_SAMPLES, 200, 3)

alpha_samp = samples[:, :, 0]   # (N_SAMPLES, 200)
beta_samp  = samples[:, :, 1]   # (N_SAMPLES, 200)

# log10(r) grid for x-axis computation
log10r_grid = sample_x.squeeze().numpy()   # (200,)
r_grid      = 10 ** log10r_grid            # actual ratio for x-axis

# HH prediction for each sample: pH = α + β·log10(r)
# log10r_grid shape (200,) broadcasts with (N_SAMPLES, 200)
pH_samples = alpha_samp.numpy() + beta_samp.numpy() * log10r_grid   # (N_SAMPLES, 200)

# Mean and std of the pH bundle
pH_mean = pH_samples.mean(axis=0)   # (200,)
pH_std  = pH_samples.std(axis=0)    # (200,)

# Global HH fit (single best-fit line over all 4 points)
alpha_global = objective_alpha(train_x_joint, train_y_ph)
beta_global  = objective_beta(train_x_joint,  train_y_ph)
pH_global    = alpha_global + beta_global * log10r_grid

# ─── Experimental data ────────────────────────────────────────────────────────
exp_r   = (valid_data_fit['vol_acid'] / valid_data_fit['vol_sodiumHydroxide']).values
exp_pH  = valid_data_fit['pH_measured'].values
exp_cell = valid_data_fit['cell'].values

# ─── Figure ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

# GP mean ± std bands
ax.plot(r_grid, pH_mean, color='#2196F3', lw=2.5,
        label='GP mean prediction', zorder=4)
ax.fill_between(r_grid,
                pH_mean - 2 * pH_std,
                pH_mean + 2 * pH_std,
                alpha=0.18, color='#2196F3', label='±2σ (95% interval)')
ax.fill_between(r_grid,
                pH_mean - pH_std,
                pH_mean + pH_std,
                alpha=0.30, color='#2196F3', label='±1σ (68% interval)')

# Global least-squares HH fit
ax.plot(r_grid, pH_global, color='#FF9800', lw=2, ls='--',
        label=f'Global fit: pH = {alpha_global:.3f} + {beta_global:.3f}·log₁₀(r)',
        zorder=3)

# Experimental data points
ax.scatter(exp_r, exp_pH, c='red', s=120, zorder=6,
           edgecolors='k', linewidths=1, marker='D',
           label='Experimental data')
for i in range(len(exp_r)):
    ax.annotate(f"pt{int(exp_cell[i])}\npH={exp_pH[i]:.2f}",
                xy=(exp_r[i], exp_pH[i]),
                xytext=(exp_r[i] * 1.15, exp_pH[i] + 0.3),
                fontsize=8.5, ha='left',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9),
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75))

# Formatting
ax.set_xscale('log')
ax.set_xlabel("V_acid / V_base  (log scale)", fontsize=12)
ax.set_ylabel("pH", fontsize=12)
ax.set_title("Henderson-Hasselbalch Function\nGP Prediction with Uncertainty vs Experimental Data",
             fontsize=13, fontweight='bold')
ax.set_ylim([0, 14])
ax.axhline(7, color='gray', lw=1, ls=':', alpha=0.5, label='Neutral pH = 7')
ax.legend(fontsize=9.5, loc='upper right')
ax.grid(True, which='both', alpha=0.25)

# Equation annotation
eq_text = (f"pH = α + β·log₁₀(r)\n"
           f"α = {alpha_global:.4f}  (intercept)\n"
           f"β = {beta_global:.4f}  (slope)")
ax.text(0.03, 0.97, eq_text, transform=ax.transAxes,
        fontsize=9.5, va='top', family='monospace',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.85))

plt.tight_layout()
plt.savefig("HH_curve.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → HH_curve.png")
print(f"Global fit: α = {alpha_global:.4f}, β = {beta_global:.4f}")

In [ ]:
# ─── GP posterior for E_pitt across input range ───────────────────────────────
N_SAMPLES = 2000
torch.manual_seed(42)

model_joint.eval()
likelihood_joint.eval()

# Dense grid over log10(ratio) range
sample_x    = torch.linspace(float(bounds[0]), float(bounds[1]),
                              200, dtype=torch.double).unsqueeze(1)  # (200, 1)
log10r_grid = sample_x.squeeze().numpy()
r_grid      = 10 ** log10r_grid

# Draw posterior samples for E_pitt (task index 2)
with torch.no_grad():
    post       = model_joint(sample_x)
    samples    = post.sample(torch.Size([N_SAMPLES]))  # (N_SAMPLES, 200, 3)

epit_samples = samples[:, :, 2].numpy()   # (N_SAMPLES, 200)
epit_mean    = epit_samples.mean(axis=0)  # (200,)
epit_std     = epit_samples.std(axis=0)   # (200,)

# ─── Experimental data ────────────────────────────────────────────────────────
exp_r    = (valid_data_fit['vol_acid'] / valid_data_fit['vol_sodiumHydroxide']).values
exp_epit = valid_data_fit['epit'].values
exp_cell = valid_data_fit['cell'].values

# ─── Figure ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

# GP mean ± std bands
ax.plot(r_grid, epit_mean, color='#E53935', lw=2.5,
        label='GP mean prediction', zorder=4)
ax.fill_between(r_grid,
                epit_mean - 2 * epit_std,
                epit_mean + 2 * epit_std,
                alpha=0.18, color='#E53935', label='±2σ (95% interval)')
ax.fill_between(r_grid,
                epit_mean - epit_std,
                epit_mean + epit_std,
                alpha=0.30, color='#E53935', label='±1σ (68% interval)')

# Experimental data points
ax.scatter(exp_r, exp_epit, c='#1565C0', s=120, zorder=6,
           edgecolors='k', linewidths=1, marker='D',
           label='Experimental data')
for i in range(len(exp_r)):
    ax.annotate(f"pt{int(exp_cell[i])}\nE={exp_epit[i]:.3f}V",
                xy=(exp_r[i], exp_epit[i]),
                xytext=(exp_r[i] * 1.15, exp_epit[i] + 0.01),
                fontsize=8.5, ha='left',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9),
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75))

# Formatting
ax.set_xscale('log')
ax.set_xlabel("V_acid / V_base  (log scale)", fontsize=12)
ax.set_ylabel("E_pitt  (V vs ref)", fontsize=12)
ax.set_title("Pitting Corrosion Potential E_pitt\nGP Surrogate Prediction vs Experimental Data",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9.5, loc='upper left')
ax.grid(True, which='both', alpha=0.25)

# Stats annotation
epit_obs_mean = exp_epit.mean()
epit_obs_std  = exp_epit.std()
stats_text = (f"Observed E_pitt\n"
              f"mean = {epit_obs_mean:.4f} V\n"
              f"std  = {epit_obs_std:.4f} V\n"
              f"range = [{exp_epit.min():.3f}, {exp_epit.max():.3f}] V")
ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
        fontsize=9, va='top', ha='right', family='monospace',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.85))

# Uncertainty flatness check — E_pitt goal is uniform σ across input range
std_cv = epit_std.std() / epit_std.mean() * 100
flat_status = f"σ uniformity CV = {std_cv:.1f}%\n{'flat ✓' if std_cv < 20 else 'not yet flat — explore more'}"
ax.text(0.03, 0.03, flat_status, transform=ax.transAxes,
        fontsize=9, va='bottom',
        color='green' if std_cv < 20 else 'red',
        bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.tight_layout()
plt.savefig("Epitt_surrogate.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → Epitt_surrogate.png")
print(f"E_pitt uncertainty uniformity CV: {std_cv:.1f}%")

## Acquisition Function

In [ ]:
class JointGPBoTorchWrapper(Model):
    _num_outputs = 3

    def __init__(self, gp_model, likelihood):
        super().__init__()
        self.model      = gp_model
        self.likelihood = likelihood

    def posterior(self, X, observation_noise=True, **kwargs):
        self.model.eval()
        self.likelihood.eval()

        # ── KEY FIX: no torch.no_grad() here ──────────────────────────────
        # optimize_acqf uses L-BFGS-B which needs gradients of the acquisition
        # score w.r.t. the candidate inputs. torch.no_grad() severs the
        # computation graph and causes the RuntimeError you saw.
        # Only use no_grad() when you're doing pure inference/plotting.
        with gpytorch.settings.fast_pred_var():
            dist = (self.likelihood(self.model(X))
                    if observation_noise
                    else self.model(X))

        return GPyTorchPosterior(dist)

    def forward(self, X):
        return self.model(X)
from botorch.acquisition.multi_objective.monte_carlo import (
    qNoisyExpectedHypervolumeImprovement
)
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning
)
from botorch.sampling.normal import SobolQMCNormalSampler

In [ ]:
# ─── Recompute bounds from physical acid ratio constraint ─────────────────────
# Individual acid ratio = vol_acid_each / vol_sodiumHydroxide
#                       = vol_ratio / 3   (since vol_acid_total = 3 * vol_acid_each)
#
# Constraint: 0.02 ≤ Va_each/Vb ≤ 12
# → 0.06 ≤ vol_ratio ≤ 36
# → log10(0.06) ≤ log10(vol_ratio) ≤ log10(36)

acid_ratio_min = 0.02
acid_ratio_max = 12.0

vol_ratio_min  = acid_ratio_min * 3    # 0.06
vol_ratio_max  = acid_ratio_max * 3    # 36.0

log_min = np.log10(vol_ratio_min)      # ~-1.222
log_max = np.log10(vol_ratio_max)      # ~1.556

bounds  = torch.tensor([[log_min], [log_max]], dtype=torch.double)

print(f"Individual acid ratio constraint: [{acid_ratio_min}, {acid_ratio_max}]")
print(f"Implied vol_ratio range:          [{vol_ratio_min:.3f}, {vol_ratio_max:.3f}]")
print(f"Bounds in log10 space:            [{log_min:.4f}, {log_max:.4f}]")

In [ ]:
# ─── Segment-constrained sequential qNEHVI ────────────────────────────────────
# The acquisition landscape is flat/monotone within the new physical bounds,
# causing all 4 steps to collapse to the same boundary point.
# Fix: divide [log_min, log_max] into 4 equal segments and optimise
# within each — this guarantees diverse coverage while still picking
# the best point within each region of the input space.

botorch_model = JointGPBoTorchWrapper(model_joint, likelihood_joint)

RANDOM_SEED = random_seed
candidates   = []

# Divide input range into 4 equal segments
segment_edges = np.linspace(log_min, log_max, 5)   # 5 edges → 4 segments
segments = [(segment_edges[i], segment_edges[i+1]) for i in range(4)]

print("Search segments (log10 scale → vol_ratio → Va_each/Vb):")
for i, (lo, hi) in enumerate(segments):
    print(f"  Segment {i+1}: log10 [{lo:.3f}, {hi:.3f}] "
          f"→ ratio [{10**lo:.3f}, {10**hi:.3f}] "
          f"→ Va/Vb [{10**lo/3:.3f}, {10**hi/3:.3f}]")

print()

for q_idx, (seg_lo, seg_hi) in enumerate(segments):
    torch.manual_seed(RANDOM_SEED + q_idx)

    sampler = SobolQMCNormalSampler(
        sample_shape=torch.Size([128]),
        seed=RANDOM_SEED + q_idx
    )

    # Baseline grows with previously selected points
    X_baseline = (train_x_joint if len(candidates) == 0
                  else torch.cat([
                      train_x_joint,
                      torch.tensor(candidates, dtype=torch.double).unsqueeze(1)
                  ], dim=0))

    qNEHVI = qNoisyExpectedHypervolumeImprovement(
        model          = botorch_model,
        ref_point      = ref_point,
        X_baseline     = X_baseline,
        sampler        = sampler,
        prune_baseline = False,
        cache_root     = False,
    )

    # Bounds restricted to this segment
    seg_bounds = torch.tensor([[seg_lo], [seg_hi]], dtype=torch.double)

    torch.manual_seed(RANDOM_SEED + q_idx)
    candidate, acq_val = optimize_acqf(
        acq_function = qNEHVI,
        bounds       = seg_bounds,      # ← segment-specific bounds
        q            = 1,
        num_restarts = 20,
        raw_samples  = 256,
        options      = {"batch_limit": 5, "maxiter": 200},
    )

    chosen = candidate.item()
    candidates.append(chosen)

    vr  = 10 ** chosen
    iar = vr / 3
    print(f"Segment {q_idx+1} [{10**seg_lo/3:.3f}, {10**seg_hi/3:.3f}] → "
          f"log10(r)={chosen:.4f}, vol_ratio={vr:.4f}, "
          f"Va_each/Vb={iar:.4f}, EHVI={acq_val.item():.6f}")


In [ ]:
# ─── Convert to lab volumes ───────────────────────────────────────────────────
TOTAL_VOL = 60.0
WATER_VOL = 10.0

def log10ratio_to_volumes(log10_ratio):
    vol_ratio   = 10 ** log10_ratio
    working_vol = TOTAL_VOL - WATER_VOL
    v_base      = working_vol / (1 + vol_ratio)
    v_acid_each = (working_vol - v_base) / 3
    return {
        'log10_vol_ratio':     round(log10_ratio, 4),
        'vol_ratio':           round(vol_ratio, 4),
        'Va_each/Vb':          round(v_acid_each / max(v_base, 1e-6), 4),
        'vol_aceticAcid':      round(max(v_acid_each, 0), 2),
        'vol_phosphoricAcid':  round(max(v_acid_each, 0), 2),
        'vol_boricAcid':       round(max(v_acid_each, 0), 2),
        'vol_sodiumHydroxide': round(max(v_base, 0), 2),
        'vol_water':           round(WATER_VOL, 2),
    }

recs   = [log10ratio_to_volumes(c) for c in candidates]
rec_df = pd.DataFrame(recs)

rec_df['total_vol'] = (rec_df['vol_aceticAcid'] +
                       rec_df['vol_phosphoricAcid'] +
                       rec_df['vol_boricAcid'] +
                       rec_df['vol_sodiumHydroxide'] +
                       rec_df['vol_water'])

in_range = rec_df['Va_each/Vb'].between(acid_ratio_min, acid_ratio_max)

print("\n" + "═" * 90)
print("RECOMMENDED NEXT 4 SAMPLING POINTS")
print("═" * 90)
print(rec_df.to_string(index=False))
print(f"\nAll volumes sum to {TOTAL_VOL} mL:             {(rec_df['total_vol'] == TOTAL_VOL).all()}")
print(f"All Va_each/Vb within [{acid_ratio_min}, {acid_ratio_max}]:  {in_range.all()}")

rec_df.drop(columns='total_vol').to_csv("recommended_points_ehvi.csv", index=False)
print("Saved → recommended_points_ehvi.csv")
#return table of ratios of each acid to base for next 4 points to test, with columns:


In [ ]:
def each_acid_to_base_ratio(csv_file):
    data = pd.read_csv(csv_file)
    ratios = pd.DataFrame()

    for i in range(len(data)):
        ratios.loc[i, "Va_AA/Vb"] = data.loc[i, 'vol_aceticAcid'] / data.loc[i, 'vol_sodiumHydroxide']
        ratios.loc[i, "Va_PA/Vb"] = data.loc[i, 'vol_phosphoricAcid'] / data.loc[i, 'vol_sodiumHydroxide']
        ratios.loc[i, "Va_BA/Vb"] = data.loc[i, 'vol_boricAcid'] / data.loc[i, 'vol_sodiumHydroxide']
    return ratios


#save the ratios to a new CSV file
ratios = each_acid_to_base_ratio("recommended_points_ehvi.csv")
ratios.to_csv("acid_to_base_ratios.csv", index=False)

print("Saved → acid_to_base_ratios.csv")
ratios

## Active Learning

In [ ]:
cr12 = pd.read_csv("combined_results-1.csv")
cr3 = pd.read_csv("combined_results-3.csv")
cr4 = pd.read_csv("combined_results-4.csv")
cr5 = pd.read_csv("combined_results-5.csv", on_bad_lines="skip")

In [ ]:
new_results = pd.concat([cr12, cr3, cr4, cr5], axis=0, ignore_index=True)
new_results["Va_BA/Vb"] = new_results['vol_boricAcid']/new_results['vol_sodiumHydroxide']
new_results

In [ ]:
# ─── Match suggested points to closest rows in results dataframe ──────────────
# Uses absolute difference in Va_each/Vb ratio as the distance metric
# since that's the physically meaningful constraint you optimised over.

# ── Inputs ────────────────────────────────────────────────────────────────────
# rec_df    : your recommended points dataframe (has 'Va_each/Vb' column)
# results   : the returned results dataframe — replace with your actual variable name

# Compute Va_each/Vb for the results dataframe if not already present
"""results['vol_acid'] = (results['vol_aceticAcid'] +
                       results['vol_phosphoricAcid'] +
                       results['vol_boricAcid'])
results['vol_ratio']    = results['vol_acid'] / results['vol_sodiumHydroxide']
results['Va_each/Vb']   = results['vol_ratio'] / 3"""

# ── Match each recommended point to its closest result row ────────────────────
matched_rows  = []
matched_idx   = []
match_summary = []

for i, rec_row in ratios.iterrows():
    target_ratio = rec_row['Va_BA/Vb']

    # Absolute difference between target and every result row
    diffs     = (new_results['Va_BA/Vb'] - target_ratio).abs()
    best_idx  = diffs.idxmin()
    best_row  = new_results.loc[best_idx]
    best_diff = diffs[best_idx]

    matched_rows.append(best_row)
    matched_idx.append(best_idx)
    match_summary.append({
        'rec_point':          i + 1,
        'rec_Va_each/Vb':     round(target_ratio, 4),
        'matched_Va_each/Vb': round(best_row['Va_BA/Vb'], 4),
        'difference':         round(best_diff, 4),
        'matched_row_idx':    best_idx,
    })

matched_df  = pd.DataFrame(matched_rows).reset_index(drop=True)
summary_df  = pd.DataFrame(match_summary)

print("═" * 65)
print("MATCH SUMMARY")
print("═" * 65)
print(summary_df.to_string(index=False))
print()
print("═" * 65)
print("MATCHED RESULT ROWS")
print("═" * 65)
print(matched_df.to_string(index=False))

# Warn if any match is poor (difference > 10% of target ratio)
for _, row in summary_df.iterrows():
    tol = row['rec_Va_each/Vb'] * 0.10
    if row['difference'] > tol:
        print(f"\n⚠  Rec point {int(row['rec_point'])}: "
              f"difference {row['difference']:.4f} exceeds 10% tolerance "
              f"({tol:.4f}) — check manually")

## Retrain Surrogate Model to include Acquisition Function Points from New Results

In [ ]:
# ─── Append matched results to existing data and retrain ──────────────────────
# matched_df contains the 4 new points returned from the lab.
# We concatenate with valid_data_fit and retrain from scratch.

# Ensure matched_df has the same columns as valid_data_fit
required_cols = ['vol_aceticAcid', 'vol_phosphoricAcid', 'vol_boricAcid',
                 'vol_sodiumHydroxide', 'pH_measured', 'epit']

# Compute vol_acid for matched rows if not present
matched_df['vol_acid'] = (matched_df['vol_aceticAcid'] +
                          matched_df['vol_phosphoricAcid'] +
                          matched_df['vol_boricAcid'])

# ─── Combine old + new data ───────────────────────────────────────────────────
valid_data_updated = pd.concat(
    [valid_data_fit, matched_df[required_cols + ['vol_acid']]],
    ignore_index=True
)

# Filter out any zero NaOH rows (same as before)
valid_data_updated = valid_data_updated[
    valid_data_updated['vol_sodiumHydroxide'] > 0
].copy()

valid_data_updated['log10_vol_ratio'] = np.log10(
    valid_data_updated['vol_acid'] / valid_data_updated['vol_sodiumHydroxide']
)

print(f"Previous dataset size: {len(valid_data_fit)}")
print(f"Updated dataset size:  {len(valid_data_updated)}")
print()
print("Full updated dataset:")
print(valid_data_updated[['pH_measured', 'vol_acid', 'vol_sodiumHydroxide',
                           'log10_vol_ratio', 'epit']].to_string(index=False))

# ─── Rebuild training tensors ─────────────────────────────────────────────────
train_x_joint = torch.tensor(
    valid_data_updated[['log10_vol_ratio']].values, dtype=torch.double
)

train_y_ph = torch.tensor(
    valid_data_updated['pH_measured'].values, dtype=torch.double
)

train_y_ab = get_alpha_beta_per_point(train_x_joint, train_y_ph)

train_y_epit = torch.tensor(
    valid_data_updated['epit'].values, dtype=torch.double
)

train_y_joint = torch.stack([
    train_y_ab[:, 0],
    train_y_ab[:, 1],
    train_y_epit
], dim=1)   # shape (n, 3)

# Sanity check
print(f"\ntrain_x_joint: {train_x_joint.T}")
print(f"train_y_joint:\n{train_y_joint}")
print(f"\nNaN in x: {train_x_joint.isnan().any().item()}")
print(f"NaN in y: {train_y_joint.isnan().any().item()}")

# Bounds — now span the full updated data range + physical constraint headroom
log_min = np.log10(0.06)   # Va_each/Vb = 0.02 → vol_ratio = 0.06
log_max = np.log10(36.0)   # Va_each/Vb = 12.0 → vol_ratio = 36.0
bounds  = torch.tensor([[log_min], [log_max]], dtype=torch.double)

print(f"\nbounds: log10(ratio) ∈ [{log_min:.3f}, {log_max:.3f}]")
print(f"  i.e. Va_each/Vb  ∈ [{10**log_min/3:.3f}, {10**log_max/3:.3f}]")

# Update valid_data_fit to point to the new combined dataset for downstream cells
valid_data_fit = valid_data_updated.copy()

In [ ]:
# ─── Retrain ──────────────────────────────────────────────────────────────────
likelihood_joint = MultitaskGaussianLikelihood(num_tasks=3).double()
model_joint      = JointGP(train_x_joint, train_y_joint, likelihood_joint).double()

model_joint.train()
likelihood_joint.train()

optimizer = torch.optim.Adam(model_joint.parameters(), lr=0.1)
mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood_joint, model_joint)

losses = []
for i in range(300):
    optimizer.zero_grad()
    loss = -mll(model_joint(train_x_joint), train_y_joint)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (i + 1) % 50 == 0:
        print(f"Iter {i+1}/300  Loss: {loss.item():.4f}")

# Loss curve
plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("Iteration"); plt.ylabel("Negative MLL")
plt.title(f"Training Loss — {len(train_x_joint)} data points")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(f"\nModel retrained on {len(train_x_joint)} points "
      f"({len(valid_data_fit) - len(matched_df)} original + "
      f"{len(matched_df)} new)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from botorch.utils.multi_objective.pareto import is_non_dominated

# ─── Split data into original and new points ──────────────────────────────────
n_orig = len(valid_data_fit) - len(matched_df)   # number of original points
n_new  = len(matched_df)                          # number of new points
n_total = len(train_x_joint)

existing_x = train_x_joint.squeeze().numpy()      # all points log10(ratio)
existing_y = train_y_joint.numpy()                # (n_total, 3)

# Labels: original points use cell number, new points labelled N1-N4
pH_labels_orig = [f"pt{int(valid_data_fit['cell'].iloc[i])}\npH={valid_data_fit['pH_measured'].iloc[i]:.1f}"
                  for i in range(n_orig)]
pH_labels_new  = [f"N{i+1}\npH={valid_data_fit['pH_measured'].iloc[n_orig + i]:.1f}"
                  for i in range(n_new)]
pH_labels_all  = pH_labels_orig + pH_labels_new

# Colors: original = blue, new = orange
face_c_orig = ['#2196F3'] * n_orig
face_c_new  = ['#FF5722'] * n_new
face_c_all  = face_c_orig + face_c_new
edge_c_all  = face_c_all

# Pareto masks
y_orig_t   = train_y_joint[:n_orig]
y_all_t    = train_y_joint
is_pareto_orig = is_non_dominated(y_orig_t).numpy()
is_pareto_all  = is_non_dominated(y_all_t).numpy()

# ─── GP posterior for plotting ────────────────────────────────────────────────
model_joint.eval()
likelihood_joint.eval()

plot_x = torch.linspace(float(bounds[0]), float(bounds[1]), 300,
                         dtype=torch.double).unsqueeze(1)
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred       = likelihood_joint(model_joint(plot_x))
    plot_mu    = pred.mean.numpy()
    lo, hi     = pred.confidence_region()
    lo, hi     = lo.numpy(), hi.numpy()

plot_x_np = plot_x.squeeze().numpy()

def norm01(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

obj_labels = ['α (normalised)', 'β (normalised)', 'E_pitt (normalised)']
obj_colors = ['#2196F3', '#4CAF50', '#FF9800']
obj_ls     = ['-', '--', '-.']


# ─── Reusable plot functions ──────────────────────────────────────────────────
def draw_input_panel(ax, x_vals, y_vals, labels, face_cols, edge_cols,
                     is_pareto_mask, title):
    """Draw input feature space panel."""
    for t, (lbl, col, ls) in enumerate(zip(obj_labels, obj_colors, obj_ls)):
        mu_n = norm01(plot_mu[:, t])
        lo_n = norm01(lo[:, t])
        hi_n = norm01(hi[:, t])
        ax.plot(plot_x_np, mu_n, color=col, lw=2, ls=ls,
                label=lbl, alpha=0.85)
        ax.fill_between(plot_x_np, lo_n, hi_n, alpha=0.12, color=col)

    for i in range(len(x_vals)):
        xi = x_vals[i]
        yi = norm01(plot_mu[:, 2])[np.argmin(np.abs(plot_x_np - xi))]
        marker = 'D' if i >= n_orig else 'o'
        ax.scatter(xi, yi, c=face_cols[i], edgecolors=edge_cols[i],
                   marker=marker, s=130, linewidths=2, zorder=6)
        if is_pareto_mask[i]:
            ax.scatter(xi, yi, marker='*', s=80, c='gold',
                       edgecolors='k', linewidths=0.5, zorder=7)
        offset_y = 0.13 if i % 2 == 0 else -0.17
        ax.annotate(labels[i],
                    xy=(xi, yi),
                    xytext=(xi + 0.05, yi + offset_y),
                    fontsize=8, ha='center',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1),
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75))

    ax.set_xlabel("log₁₀(V_acid / V_base)", fontsize=10)
    ax.set_ylabel("Normalised objective value", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlim([float(bounds[0]) - 0.1, float(bounds[1]) + 0.1])
    ax.set_ylim([-0.3, 1.45])
    ax.legend(fontsize=7.5, loc='upper right')
    ax.grid(True, alpha=0.25)


def draw_response_panel(ax, y_vals, labels, face_cols, edge_cols,
                        is_pareto_mask, x_idx, y_idx, xlabel, ylabel, title):
    """Draw one response space panel."""
    ax.set_title(title, fontsize=9.5, fontweight='bold')
    for i in range(len(y_vals)):
        alpha  = 1.0 if is_pareto_mask[i] else 0.35
        marker = 'D' if i >= n_orig else 'o'
        ax.scatter(y_vals[i, x_idx], y_vals[i, y_idx],
                   c=face_cols[i], edgecolors=edge_cols[i],
                   marker=marker, s=100, linewidths=2,
                   alpha=alpha, zorder=5)
        if is_pareto_mask[i]:
            ax.scatter(y_vals[i, x_idx], y_vals[i, y_idx],
                       marker='*', s=60, c='gold',
                       edgecolors='k', linewidths=0.5, zorder=7)
        x_range = max(y_vals[:, x_idx].max() - y_vals[:, x_idx].min(), 1e-6)
        y_range = max(y_vals[:, y_idx].max() - y_vals[:, y_idx].min(), 1e-6)
        ax.annotate(labels[i],
                    xy=(y_vals[i, x_idx], y_vals[i, y_idx]),
                    xytext=(y_vals[i, x_idx] + 0.03 * x_range,
                            y_vals[i, y_idx] + 0.05 * y_range),
                    fontsize=7, ha='left',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

    # Pareto staircase
    pareto_pts = y_vals[is_pareto_mask][:, [x_idx, y_idx]]
    if len(pareto_pts) > 1:
        sorted_p = pareto_pts[np.argsort(pareto_pts[:, 0])]
        ax.step(sorted_p[:, 0], sorted_p[:, 1], where='post',
                color='gold', lw=2, ls='--', alpha=0.85,
                zorder=4, label='Pareto front')
        ax.legend(fontsize=7.5)

    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(True, alpha=0.25)


# ─── Figure: side-by-side ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 13))
fig.suptitle("Pareto Frontier Evolution — Original vs Updated Dataset",
             fontsize=15, fontweight='bold', y=0.99)

# Left column = original (4 pts), Right column = updated (4 + new pts)
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.48, wspace=0.32,
                       height_ratios=[1.6, 1, 1, 1])

ax_in_L  = fig.add_subplot(gs[0, 0])
ax_ab_L  = fig.add_subplot(gs[1, 0])
ax_aep_L = fig.add_subplot(gs[2, 0])
ax_bep_L = fig.add_subplot(gs[3, 0])

ax_in_R  = fig.add_subplot(gs[0, 1])
ax_ab_R  = fig.add_subplot(gs[1, 1])
ax_aep_R = fig.add_subplot(gs[2, 1])
ax_bep_R = fig.add_subplot(gs[3, 1])

# Column headers
for ax, title in [(ax_in_L, f"ORIGINAL  ({n_orig} points)"),
                  (ax_in_R, f"UPDATED  ({n_orig} original + {n_new} new)")]:
    ax.text(0.5, 1.07, title, transform=ax.transAxes,
            fontsize=12, fontweight='bold', ha='center', va='bottom',
            color='#1A237E' if 'ORIG' in title else '#BF360C',
            bbox=dict(boxstyle='round', fc='#E3F2FD' if 'ORIG' in title
                      else '#FBE9E7', alpha=0.7))

# ── Left column: original 4 points only ──────────────────────────────────────
y_orig_np = existing_y[:n_orig]

draw_input_panel(ax_in_L,
                 existing_x[:n_orig], y_orig_np,
                 pH_labels_orig,
                 face_c_orig, face_c_orig,
                 is_pareto_orig,
                 "Input Feature Space")

draw_response_panel(ax_ab_L,  y_orig_np, pH_labels_orig,
                    face_c_orig, face_c_orig, is_pareto_orig,
                    0, 1, 'α', 'β', 'α vs β')
draw_response_panel(ax_aep_L, y_orig_np, pH_labels_orig,
                    face_c_orig, face_c_orig, is_pareto_orig,
                    0, 2, 'α', 'E_pitt (V)', 'α vs E_pitt')
draw_response_panel(ax_bep_L, y_orig_np, pH_labels_orig,
                    face_c_orig, face_c_orig, is_pareto_orig,
                    1, 2, 'β', 'E_pitt (V)', 'β vs E_pitt')

# ── Right column: all points ──────────────────────────────────────────────────
draw_input_panel(ax_in_R,
                 existing_x, existing_y,
                 pH_labels_all,
                 face_c_all, edge_c_all,
                 is_pareto_all,
                 "Input Feature Space")

draw_response_panel(ax_ab_R,  existing_y, pH_labels_all,
                    face_c_all, edge_c_all, is_pareto_all,
                    0, 1, 'α', 'β', 'α vs β')
draw_response_panel(ax_aep_R, existing_y, pH_labels_all,
                    face_c_all, edge_c_all, is_pareto_all,
                    0, 2, 'α', 'E_pitt (V)', 'α vs E_pitt')
draw_response_panel(ax_bep_R, existing_y, pH_labels_all,
                    face_c_all, edge_c_all, is_pareto_all,
                    1, 2, 'β', 'E_pitt (V)', 'β vs E_pitt')

# ── Shared legend ─────────────────────────────────────────────────────────────
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2196F3',
           markeredgecolor='#2196F3', markersize=9, label='Original point'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='#FF5722',
           markeredgecolor='#FF5722', markersize=9, label='New point'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='gold',
           markeredgecolor='k', markersize=12, label='Non-dominated (Pareto ★)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2196F3',
           markeredgecolor='#2196F3', markersize=9, alpha=0.35,
           label='Dominated (faded)'),
    Line2D([0], [0], color='gold', lw=2, ls='--', label='Pareto front'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5,
           fontsize=10, frameon=True,
           bbox_to_anchor=(0.5, -0.01))

plt.savefig("pareto_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → pareto_comparison.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.multi_objective.hypervolume import Hypervolume

# ─── Shared setup ─────────────────────────────────────────────────────────────
n_orig  = len(valid_data_fit) - len(matched_df)
n_total = len(train_x_joint)
obj_names  = ['α', 'β', 'E_pitt']
obj_colors = ['#2196F3', '#4CAF50', '#FF9800']

ref_pt  = make_ref_point(train_y_joint, slack=0.1)
hv_calc = Hypervolume(ref_point=ref_pt)

plot_x = torch.linspace(float(bounds[0]), float(bounds[1]), 300,
                         dtype=torch.double).unsqueeze(1)

# ─── Run LOO simulation for both datasets ────────────────────────────────────
def run_loo(train_x, train_y, train_y_ph_local, n_start=2):
    """
    Returns rmse_per_iter, std_per_iter, hv_per_iter, iter_labels
    for a given dataset.
    """
    n = len(train_x)
    rmse_d  = {name: [] for name in obj_names}
    std_d   = {name: [] for name in obj_names}
    hv_list = []
    labels  = []

    for k in range(n_start, n + 1):
        idx_train = list(range(k))
        idx_test  = list(range(k, n))

        x_tr = train_x[idx_train]
        y_tr = train_y[idx_train]
        x_te = train_x[idx_test]
        y_te = train_y[idx_test]

        # Retrain
        lik_k   = MultitaskGaussianLikelihood(num_tasks=3).double()
        mod_k   = JointGP(x_tr, y_tr, lik_k).double()
        mod_k.train(); lik_k.train()
        opt_k = torch.optim.Adam(mod_k.parameters(), lr=0.1)
        mll_k = gpytorch.mlls.ExactMarginalLogLikelihood(lik_k, mod_k)
        for _ in range(300):
            opt_k.zero_grad()
            loss_k = -mll_k(mod_k(x_tr), y_tr)
            loss_k.backward()
            opt_k.step()
        mod_k.eval(); lik_k.eval()

        # RMSE
        if len(idx_test) > 0:
            with torch.no_grad(), gpytorch.settings.fast_pred_var():
                mu_te = lik_k(mod_k(x_te)).mean.numpy()
            y_te_np = y_te.numpy()
            for t, name in enumerate(obj_names):
                rmse_d[name].append(
                    np.sqrt(np.mean((mu_te[:, t] - y_te_np[:, t])**2)))
        else:
            with torch.no_grad(), gpytorch.settings.fast_pred_var():
                mu_tr = lik_k(mod_k(x_tr)).mean.numpy()
            y_tr_np = y_tr.numpy()
            for t, name in enumerate(obj_names):
                rmse_d[name].append(
                    np.sqrt(np.mean((mu_tr[:, t] - y_tr_np[:, t])**2)))

        # Uncertainty
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            std_grid = lik_k(mod_k(plot_x)).variance.sqrt().numpy()
        for t, name in enumerate(obj_names):
            std_d[name].append(std_grid[:, t].mean())

        # Hypervolume
        pareto_k  = is_non_dominated(train_y[idx_train])
        hv_list.append(hv_calc.compute(train_y[idx_train][pareto_k]))
        labels.append(f"k={k}")
        print(f"  k={k}: RMSE α={rmse_d['α'][-1]:.4f}  "
              f"β={rmse_d['β'][-1]:.4f}  "
              f"E_pitt={rmse_d['E_pitt'][-1]:.4f}  "
              f"HV={hv_list[-1]:.6f}")

    return rmse_d, std_d, hv_list, labels


# Original dataset (first n_orig points)
print(f"Running LOO on original {n_orig} points...")
x_orig = train_x_joint[:n_orig]
y_orig = train_y_joint[:n_orig]
rmse_orig, std_orig, hv_orig, labels_orig = run_loo(x_orig, y_orig, None)

# Updated dataset (all points)
print(f"\nRunning LOO on updated {n_total} points...")
rmse_upd, std_upd, hv_upd, labels_upd = run_loo(train_x_joint, train_y_joint, None)


# ─── Figure 1: Predictive Accuracy — side by side ─────────────────────────────
fig1, axes1 = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
fig1.suptitle("Predictive Accuracy over Active-Learning Iterations\n"
              "Top: Original data  |  Bottom: Updated data (+ new points)",
              fontsize=13, fontweight='bold')

for row, (rmse_d, labels, tag, shade_col) in enumerate([
    (rmse_orig, labels_orig, f'Original ({n_orig} pts)', '#E3F2FD'),
    (rmse_upd,  labels_upd,  f'Updated ({n_total} pts)',  '#FBE9E7'),
]):
    iter_x = np.arange(len(labels))
    for col, (name, color) in enumerate(zip(obj_names, obj_colors)):
        ax = axes1[row, col]
        ax.plot(iter_x, rmse_d[name], 'o-',
                color=color, lw=2.5, ms=9,
                markeredgecolor='k', markeredgewidth=0.8)
        for xi, yi in zip(iter_x, rmse_d[name]):
            ax.annotate(f"{yi:.4f}", (xi, yi),
                        textcoords='offset points', xytext=(0, 10),
                        ha='center', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
        ax.set_xticks(iter_x)
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_xlabel("Training set size (k)", fontsize=10)
        ax.set_ylabel("RMSE", fontsize=10)
        ax.set_title(f"[{tag}]  Surrogate: {name}", fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.axvspan(iter_x[-1] - 0.4, iter_x[-1] + 0.4,
                   alpha=0.15, color='gray', label='Train residual\n(no held-out)')
        # Shade new-data iterations on bottom row
        if row == 1 and len(iter_x) > len(labels_orig):
            ax.axvspan(len(labels_orig) - 0.5, iter_x[-1] + 0.5,
                       alpha=0.10, color='#FF5722', label='New points')
        ax.legend(fontsize=7.5)

plt.tight_layout()
plt.savefig("model_accuracy_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


# ─── Figure 2: Uncertainty + Hypervolume — side by side ──────────────────────
fig2, axes2 = plt.subplots(2, 4, figsize=(20, 9))
fig2.suptitle("Uncertainty Evolution & Hypervolume Improvement\n"
              "Top: Original data  |  Bottom: Updated data (+ new points)",
              fontsize=13, fontweight='bold')

for row, (std_d, hv_list, labels, tag) in enumerate([
    (std_orig, hv_orig, labels_orig, f'Original ({n_orig} pts)'),
    (std_upd,  hv_upd,  labels_upd,  f'Updated ({n_total} pts)'),
]):
    iter_x = np.arange(len(labels))

    # Uncertainty panels
    for col, (name, color) in enumerate(zip(obj_names, obj_colors)):
        ax = axes2[row, col]
        ax.plot(iter_x, std_d[name], 's-',
                color=color, lw=2.5, ms=9,
                markeredgecolor='k', markeredgewidth=0.8)
        for xi, yi in zip(iter_x, std_d[name]):
            ax.annotate(f"{yi:.4f}", (xi, yi),
                        textcoords='offset points', xytext=(0, 10),
                        ha='center', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
        ax.set_xticks(iter_x)
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_xlabel("Training set size (k)", fontsize=10)
        ax.set_ylabel("Mean posterior σ", fontsize=10)
        ax.set_title(f"[{tag}]  Uncertainty: {name}", fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
        if row == 1 and len(iter_x) > len(labels_orig):
            ax.axvspan(len(labels_orig) - 0.5, iter_x[-1] + 0.5,
                       alpha=0.10, color='#FF5722')
        goal = ("Goal: ↓ converge" if name in ['α', 'β']
                else "Goal: flatten (uniform σ)")
        ax.text(0.05, 0.92, goal, transform=ax.transAxes,
                fontsize=7.5, color=color, style='italic',
                bbox=dict(boxstyle='round', fc='white', alpha=0.6))

    # Hypervolume panel
    ax   = axes2[row, 3]
    hv_arr = np.array(hv_list)
    hv_imp = np.diff(hv_arr, prepend=0.0)

    ax.bar(iter_x, hv_arr, color='#9C27B0', alpha=0.6,
           edgecolor='k', linewidth=0.8, label='Cumulative HV')
    ax.plot(iter_x, hv_arr, 'o-', color='#4A148C',
            lw=2, ms=8, markeredgecolor='k', markeredgewidth=0.8)

    ax_twin = ax.twinx()
    ax_twin.bar(iter_x, hv_imp, color='#FF5722', alpha=0.35,
                edgecolor='k', linewidth=0.8, label='HV improvement')
    ax_twin.set_ylabel("HV improvement (Δ)", fontsize=9, color='#FF5722')
    ax_twin.tick_params(axis='y', labelcolor='#FF5722')

    for xi, yi in zip(iter_x, hv_arr):
        ax.annotate(f"{yi:.4f}", (xi, yi),
                    textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=7.5,
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

    if row == 1 and len(iter_x) > len(labels_orig):
        ax.axvspan(len(labels_orig) - 0.5, iter_x[-1] + 0.5,
                   alpha=0.10, color='#FF5722', label='New points')

    ax.set_xticks(iter_x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_xlabel("Training set size (k)", fontsize=10)
    ax.set_ylabel("Hypervolume", fontsize=10)
    ax.set_title(f"[{tag}]  Hypervolume Improvement",
                 fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax_twin.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7.5, loc='upper left')

plt.tight_layout()
plt.savefig("uncertainty_hv_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → model_accuracy_comparison.png, uncertainty_hv_comparison.png")

In [ ]:
from scipy.stats import gaussian_kde

N_SAMPLES = 2000
n_orig    = len(valid_data_fit) - len(matched_df)

# ─── Helper: draw samples from a trained model ────────────────────────────────
def sample_posterior(model, x_grid, n_samples, seed):
    torch.manual_seed(seed)
    model.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        post    = model(x_grid)
        samples = post.sample(torch.Size([n_samples]))  # (n_samples, grid, 3)
    return samples.numpy()

sample_x = torch.linspace(float(bounds[0]), float(bounds[1]),
                           100, dtype=torch.double).unsqueeze(1)

# ─── Train model on original data only ───────────────────────────────────────
x_orig = train_x_joint[:n_orig]
y_orig = train_y_joint[:n_orig]
y_ph_orig = train_y_ph[:n_orig]

lik_orig   = MultitaskGaussianLikelihood(num_tasks=3).double()
mod_orig   = JointGP(x_orig, y_orig, lik_orig).double()
mod_orig.train(); lik_orig.train()
opt_orig = torch.optim.Adam(mod_orig.parameters(), lr=0.1)
mll_orig = gpytorch.mlls.ExactMarginalLogLikelihood(lik_orig, mod_orig)
for i in range(300):
    opt_orig.zero_grad()
    loss = -mll_orig(mod_orig(x_orig), y_orig)
    loss.backward()
    opt_orig.step()
print("Original model trained.")

# ─── Sample from both models ──────────────────────────────────────────────────
samp_orig = sample_posterior(mod_orig,   sample_x, N_SAMPLES, random_seed)
samp_upd  = sample_posterior(model_joint, sample_x, N_SAMPLES, random_seed)

# Global fits
alpha_global_orig = objective_alpha(x_orig,          y_ph_orig)
beta_global_orig  = objective_beta(x_orig,            y_ph_orig)
alpha_global_upd  = objective_alpha(train_x_joint,   train_y_ph)
beta_global_upd   = objective_beta(train_x_joint,    train_y_ph)

# LOO estimates
alpha_loo_orig = y_orig[:, 0].numpy()
beta_loo_orig  = y_orig[:, 1].numpy()
alpha_loo_upd  = train_y_joint[:, 0].numpy()
beta_loo_upd   = train_y_joint[:, 1].numpy()

# ─── Figure: 2 rows × 2 cols  (row = orig/updated, col = α/β) ────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Histogram of Predicted α and β — Convergence Comparison\n"
             f"({N_SAMPLES} posterior samples × 100 input locations)",
             fontsize=13, fontweight='bold')

datasets = [
    # (samples,      loo_alpha,        loo_beta,       a_glob,           b_glob,          row_tag,                    pt_prefix)
    (samp_orig, alpha_loo_orig, beta_loo_orig, alpha_global_orig, beta_global_orig,
     f'Original ({n_orig} pts)',  'pt'),
    (samp_upd,  alpha_loo_upd,  beta_loo_upd,  alpha_global_upd,  beta_global_upd,
     f'Updated ({len(train_x_joint)} pts)', 'pt'),
]

param_meta = [
    ('α (intercept)', '#2196F3', 0),
    ('β (slope)',     '#4CAF50', 1),
]

for row, (samp, a_loo, b_loo, a_glob, b_glob, row_tag, prefix) in enumerate(datasets):
    for col, (name, color, task_idx) in enumerate(param_meta):

        ax        = axes[row, col]
        loo_vals  = a_loo if task_idx == 0 else b_loo
        glob_val  = a_glob if task_idx == 0 else b_glob
        samp_flat = samp[:, :, task_idx].ravel()

        # Histogram
        ax.hist(samp_flat, bins=60, color=color, edgecolor='white',
                alpha=0.70, density=True, label='GP posterior samples')

        # KDE
        kde = gaussian_kde(samp_flat)
        xs  = np.linspace(samp_flat.min(), samp_flat.max(), 300)
        ax.plot(xs, kde(xs), color=color, lw=2.5, label='KDE')

        # Global fit
        ax.axvline(glob_val, color='red', lw=2.5, ls='-',
                   label=f'Global fit = {glob_val:.4f}')

        # LOO estimates
        for i, v in enumerate(loo_vals):
            ax.axvline(v, color='orange', lw=1.5, ls='--',
                       label='LOO estimates' if i == 0 else '')
            ax.text(v, ax.get_ylim()[1] * 0.02, f' {prefix}{i+1}',
                    color='darkorange', fontsize=7.5, rotation=90, va='bottom')

        # Posterior mean ± 1σ
        mu_s  = samp_flat.mean()
        std_s = samp_flat.std()
        ax.axvline(mu_s, color='black', lw=1.5, ls=':',
                   label=f'Posterior mean = {mu_s:.4f}')
        ax.axvspan(mu_s - std_s, mu_s + std_s,
                   alpha=0.12, color='black', label=f'±1σ = {std_s:.4f}')

        ax.set_xlabel(name, fontsize=11)
        ax.set_ylabel("Density", fontsize=11)
        ax.set_title(f"[{row_tag}]  {name}\nmean={mu_s:.4f}, std={std_s:.4f}",
                     fontsize=10, fontweight='bold')
        ax.legend(fontsize=7.5, loc='upper right')
        ax.grid(True, alpha=0.25)

        # CV convergence check
        cv        = abs(std_s / mu_s) * 100
        converged = cv < 5.0
        status    = f"CV = {cv:.1f}% → {'converged ✓' if converged else 'not yet converged'}"
        ax.text(0.03, 0.97, status, transform=ax.transAxes,
                fontsize=8.5, va='top',
                color='green' if converged else 'red',
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))

        # Row background tint
        ax.set_facecolor('#F3F8FF' if row == 0 else '#FFF5F3')

        ax.relim()
        ax.autoscale_view()

# Row labels on the left
for row, tag in enumerate([f'ORIGINAL ({n_orig} pts)',
                             f'UPDATED ({len(train_x_joint)} pts)']):
    axes[row, 0].annotate(tag, xy=(-0.18, 0.5),
                          xycoords='axes fraction',
                          fontsize=10, fontweight='bold', va='center',
                          rotation=90,
                          color='#1565C0' if row == 0 else '#BF360C')

plt.tight_layout()
plt.savefig("alpha_beta_histograms_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → alpha_beta_histograms_comparison.png")

In [ ]:
# ─── HH curve comparison (1 row × 2 cols) ────────────────────────────────────
N_SAMPLES = 2000
n_orig    = len(valid_data_fit) - len(matched_df)

sample_x    = torch.linspace(float(bounds[0]), float(bounds[1]),
                              100, dtype=torch.double).unsqueeze(1)
log10r_grid = sample_x.squeeze().numpy()
r_grid      = 10 ** log10r_grid

def hh_bundle(model, seed):
    torch.manual_seed(seed)
    model.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        samples = model(sample_x).sample(torch.Size([N_SAMPLES])).numpy()
    pH = samples[:, :, 0] + samples[:, :, 1] * log10r_grid
    return pH.mean(axis=0), pH.std(axis=0)

pH_mean_orig, pH_std_orig = hh_bundle(mod_orig,    random_seed)
pH_mean_upd,  pH_std_upd  = hh_bundle(model_joint, random_seed)

alpha_global_orig = objective_alpha(x_orig,        y_ph_orig)
beta_global_orig  = objective_beta(x_orig,         y_ph_orig)
alpha_global_upd  = objective_alpha(train_x_joint, train_y_ph)
beta_global_upd   = objective_beta(train_x_joint,  train_y_ph)

exp_r_orig  = (valid_data_fit['vol_acid'].iloc[:n_orig] /
               valid_data_fit['vol_sodiumHydroxide'].iloc[:n_orig]).values
exp_pH_orig = valid_data_fit['pH_measured'].iloc[:n_orig].values
exp_r_upd   = (valid_data_fit['vol_acid'] /
               valid_data_fit['vol_sodiumHydroxide']).values
exp_pH_upd  = valid_data_fit['pH_measured'].values

# ─── Figure ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle("Henderson-Hasselbalch Function — Original vs Updated\n"
             "GP Prediction with Uncertainty vs Experimental Data",
             fontsize=13, fontweight='bold')

hh_datasets = [
    (pH_mean_orig, pH_std_orig, alpha_global_orig, beta_global_orig,
     exp_r_orig, exp_pH_orig, f'Original ({n_orig} pts)', '#F3F8FF'),
    (pH_mean_upd,  pH_std_upd,  alpha_global_upd,  beta_global_upd,
     exp_r_upd,  exp_pH_upd,  f'Updated ({len(train_x_joint)} pts)', '#FFF5F3'),
]

for ax, (pH_mean, pH_std, a_g, b_g,
         exp_r, exp_pH, tag, bg) in zip(axes, hh_datasets):

    ax.plot(r_grid, pH_mean, color='#2196F3', lw=2.5, label='GP mean', zorder=4)
    ax.fill_between(r_grid, pH_mean - 2*pH_std, pH_mean + 2*pH_std,
                    alpha=0.15, color='#2196F3', label='±2σ')
    ax.fill_between(r_grid, pH_mean - pH_std,   pH_mean + pH_std,
                    alpha=0.28, color='#2196F3', label='±1σ')
    ax.plot(r_grid, a_g + b_g * log10r_grid,
            color='#FF9800', lw=2, ls='--',
            label=f'Global fit: α={a_g:.3f}, β={b_g:.3f}')
    ax.scatter(exp_r, exp_pH, c='red', s=100, zorder=6,
               edgecolors='k', linewidths=1, marker='D',
               label='Experimental data')
    for i in range(len(exp_r)):
        ax.annotate(f"pt{i+1}\npH={exp_pH[i]:.2f}",
                    xy=(exp_r[i], exp_pH[i]),
                    xytext=(exp_r[i] * 1.15, exp_pH[i] + 0.3),
                    fontsize=8, ha='left',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8),
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
    ax.set_xscale('log')
    ax.set_xlabel("V_acid / V_base  (log scale)", fontsize=11)
    ax.set_ylabel("pH", fontsize=11)
    ax.set_title(f"[{tag}]  Henderson-Hasselbalch Curve",
                 fontsize=11, fontweight='bold')
    ax.set_ylim([0, 14])
    ax.axhline(7, color='gray', lw=1, ls=':', alpha=0.5, label='Neutral pH = 7')
    ax.legend(fontsize=8.5, loc='upper right')
    ax.grid(True, which='both', alpha=0.25)
    ax.set_facecolor(bg)
    ax.text(0.03, 0.05, f"pH = {a_g:.3f} + {b_g:.3f}·log₁₀(r)",
            transform=ax.transAxes, fontsize=9, family='monospace',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.85))

plt.tight_layout()
plt.savefig("HH_curve_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → HH_curve_comparison.png")

In [ ]:
# ─── E_pitt surrogate comparison — Original vs Updated ────────────────────────
N_SAMPLES = 2000
n_orig    = len(valid_data_fit) - len(matched_df)

sample_x    = torch.linspace(float(bounds[0]), float(bounds[1]),
                              200, dtype=torch.double).unsqueeze(1)
log10r_grid = sample_x.squeeze().numpy()
r_grid      = 10 ** log10r_grid

def epit_bundle(model, seed):
    torch.manual_seed(seed)
    model.eval()
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        samples = model(sample_x).sample(torch.Size([N_SAMPLES])).numpy()
    epit = samples[:, :, 2]
    return epit.mean(axis=0), epit.std(axis=0)

epit_mean_orig, epit_std_orig = epit_bundle(mod_orig,    random_seed)
epit_mean_upd,  epit_std_upd  = epit_bundle(model_joint, random_seed)

# Experimental data split
exp_r_orig    = (valid_data_fit['vol_acid'].iloc[:n_orig] /
                 valid_data_fit['vol_sodiumHydroxide'].iloc[:n_orig]).values
exp_epit_orig = valid_data_fit['epit'].iloc[:n_orig].values

exp_r_upd     = (valid_data_fit['vol_acid'] /
                 valid_data_fit['vol_sodiumHydroxide']).values
exp_epit_upd  = valid_data_fit['epit'].values

# ─── Build point labels safely (handle missing cell column) ───────────────────
def make_labels(df, n_orig_pts):
    labels = []
    for i in range(len(df)):
        if i < n_orig_pts and 'cell' in df.columns and not pd.isna(df['cell'].iloc[i]):
            labels.append(f"pt{int(df['cell'].iloc[i])}")
        else:
            labels.append(f"N{i - n_orig_pts + 1}" if i >= n_orig_pts else f"pt{i+1}")
    return labels

labels_orig = make_labels(valid_data_fit.iloc[:n_orig], n_orig)
labels_upd  = make_labels(valid_data_fit, n_orig)

# ─── Figure ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle("Pitting Corrosion Potential E_pitt — Original vs Updated\n"
             "GP Surrogate Prediction vs Experimental Data",
             fontsize=13, fontweight='bold')

epit_datasets = [
    (epit_mean_orig, epit_std_orig,
     exp_r_orig, exp_epit_orig, labels_orig,
     f'Original ({n_orig} pts)', '#F3F8FF'),
    (epit_mean_upd,  epit_std_upd,
     exp_r_upd,  exp_epit_upd,  labels_upd,
     f'Updated ({len(train_x_joint)} pts)', '#FFF5F3'),
]

for ax, (epit_mean, epit_std,
         exp_r, exp_epit, pt_labels,
         tag, bg) in zip(axes, epit_datasets):

    # GP mean ± bands
    ax.plot(r_grid, epit_mean, color='#E53935', lw=2.5,
            label='GP mean prediction', zorder=4)
    ax.fill_between(r_grid,
                    epit_mean - 2 * epit_std,
                    epit_mean + 2 * epit_std,
                    alpha=0.18, color='#E53935', label='±2σ (95% interval)')
    ax.fill_between(r_grid,
                    epit_mean - epit_std,
                    epit_mean + epit_std,
                    alpha=0.30, color='#E53935', label='±1σ (68% interval)')

    # Original points
    ax.scatter(exp_r[:n_orig], exp_epit[:n_orig],
               c='#1565C0', s=120, zorder=6,
               edgecolors='k', linewidths=1, marker='D',
               label=f'Original data ({n_orig} pts)')

    # New points (right panel only)
    if len(exp_r) > n_orig:
        ax.scatter(exp_r[n_orig:], exp_epit[n_orig:],
                   c='#FF5722', s=120, zorder=6,
                   edgecolors='k', linewidths=1, marker='D',
                   label=f'New data ({len(exp_r) - n_orig} pts)')

    # Annotations — label is now always a clean string
    for i in range(len(exp_r)):
        color_ann = '#1565C0' if i < n_orig else '#FF5722'
        ax.annotate(f"{pt_labels[i]}\nE={exp_epit[i]:.3f}V",
                    xy=(exp_r[i], exp_epit[i]),
                    xytext=(exp_r[i] * 1.15, exp_epit[i] + 0.01),
                    fontsize=8, ha='left', color=color_ann,
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8),
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

    ax.set_xscale('log')
    ax.set_xlabel("V_acid / V_base  (log scale)", fontsize=11)
    ax.set_ylabel("E_pitt  (V vs ref)", fontsize=11)
    ax.set_title(f"[{tag}]  E_pitt Surrogate",
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8.5, loc='upper left')
    ax.grid(True, which='both', alpha=0.25)
    ax.set_facecolor(bg)

    stats_text = (f"Observed E_pitt\n"
                  f"mean  = {exp_epit.mean():.4f} V\n"
                  f"std   = {exp_epit.std():.4f} V\n"
                  f"range = [{exp_epit.min():.3f}, {exp_epit.max():.3f}] V")
    ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
            fontsize=8.5, va='top', ha='right', family='monospace',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.85))

    std_cv = epit_std.std() / epit_std.mean() * 100
    flat_status = (f"σ uniformity CV = {std_cv:.1f}%\n"
                   f"{'flat ✓' if std_cv < 20 else 'not yet flat — explore more'}")
    ax.text(0.03, 0.03, flat_status, transform=ax.transAxes,
            fontsize=8.5, va='bottom',
            color='green' if std_cv < 20 else 'red',
            bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.tight_layout()
plt.savefig("Epitt_surrogate_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → Epitt_surrogate_comparison.png")

In [ ]:
#add markdowns